# 🔭 Laboratorio: Macro-Backtesting de Validación Secuencial
**Objetivo:** Ejecutar simulaciones `Walk-Forward` masivas sobre múltiples activos y marcos temporales para crear una base de datos de rendimiento empírico del motor SINDy.
**Métricas:** RMSE y Hit Ratio (Direccionalidad).
**Archivo de Salida:** `macro_backtest_db.csv` (Anexión Segura / Checkpointing).

In [5]:
import sys
import os
import time
import warnings
sys.path.append(os.path.abspath('..'))

# Ocultar warnings matemáticos durante la simulación masiva
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from src.ui.market_loader import MarketLoader
from src.quant_engine.evaluator import WalkForwardEvaluator

### 1. Configuración de la Batería de Pruebas
Aquí se define el universo de activos y las temporalidades a auditar.

In [ ]:
# 1. Definir los Tickers a analizar
tickers = ['MSFT', 'XLF', 'C', 'BTC-USD', 'SPY', 'AAPL', 'ETH-USD', 'JPM', 'BAC', 'MA', 'AMZN' ]

# 2. Definir los tamaños de Ventana de Contexto (lookback en velas diarias)
# 60 ≈ 3 meses, 120 ≈ 6 meses, 252 ≈ 1 año, 500 ≈ 2 años, 750 ≈ 3 años, 1250 ≈ 5 años, 0 = Todo el historial (Legacy)
context_windows = [1250]

db_path = "context_optimization_db.csv"

# 3. Manejador de DB y Checkpointing (Idempotencia)
processed_configs = set()
if os.path.exists(db_path):
    df_existente = pd.read_csv(db_path)
    if not df_existente.empty:
        # Agrupar por Ticker y Context_Window para saber qué combinaciones ya terminaron
        agrupado = df_existente.groupby(['Ticker', 'Context_Window']).size().reset_index()
        for _, row in agrupado.iterrows():
            processed_configs.add((row['Ticker'], int(row['Context_Window'])))

print(f"⚙️ Total Combinaciones Posibles en el Universo: {len(tickers) * len(context_windows)}")
print(f"📂 Combinaciones ya procesadas y guardadas en DB: {len(processed_configs)}\n")


⚙️ Total Combinaciones Posibles en el Universo: 6
📂 Combinaciones ya procesadas y guardadas en DB: 0



### 2. Motor de Ejecución Masiva
Este proceso puede tardar horas. Si lo detienes, la próxima vez que lo inicies continuará donde se quedó.

In [7]:
for ticker in tickers:
    for cw in context_windows:
        config_key = (ticker, cw)
        
        if config_key in processed_configs:
            print(f"⏭️ Omitiendo {ticker} [Contexto: {cw}] - Ya existe en la base de datos.")
            continue
            
        print(f"\n🚀 Iniciando simulación para {ticker} [Contexto: {cw}]...")
        
        # --- DESCARGA CON PROTECCIÓN ---
        try:
            df_mercado = MarketLoader.load_ticker_data(ticker, period="10y", interval="1d")
        except Exception as e:
            print(f"❌ Error descargando datos para {ticker}: {e}")
            continue
            
        total_velas = len(df_mercado)
        if total_velas < 200:
            print(f"⚠️ Omitiendo {ticker}: Historial demasiado corto ({total_velas} velas).")
            continue
            
        # --- LÓGICA DE SALTOS DINÁMICOS ---
        SALTO = 20
        VENTANA_INICIAL = 150
        HORIZONTE = 300
        BLOQUES = 60
        
        print(f"   ► Velas Disponibles: {total_velas} | Salto Iterativo: {SALTO} | Predicción a Ciegas: {HORIZONTE}")
        
        # Instanciamos el Motor con la ventana de contexto
        evaluador = WalkForwardEvaluator(df_mercado, disable_norm=False, disable_returns=False, context_window=cw)
        
        try:
            # Ejecutar el Auto-Tuner Iterativo en el tiempo
            df_resultados = evaluador.run(initial_window=VENTANA_INICIAL, stride=SALTO, horizon=HORIZONTE, blocks=BLOQUES)
            
            # Extraer resultados brutos
            df_guardar = df_resultados.copy()
            
            # INYECCIÓN DE METADATOS PARA EL FUTURO QUERY NOTEBOOK
            df_guardar.insert(0, 'Context_Window', cw)
            df_guardar.insert(0, 'Total_Velas_Disponible', total_velas)
            df_guardar.insert(0, 'Intervalo_Velas', '1d')
            df_guardar.insert(0, 'Periodo_Historia', '10y')
            df_guardar.insert(0, 'Ticker', ticker)
            
            # Checkpoint Progressivo al CSV
            file_exists = os.path.isfile(db_path)
            df_guardar.to_csv(db_path, mode='a', header=not file_exists, index=False)
            
            print(f"✅ Éxito. Guardadas {len(df_guardar)} iteraciones para {ticker} en {db_path}.")
            
            # Registrar para que no se repita en caso de caída posterior en este mismo loop
            processed_configs.add(config_key)
            
            # Breve pausa para limpiar I/O
            time.sleep(1)
            
        except Exception as e:
            print(f"❌ Error matemático/sistémico durante el Backtest de {ticker}: {e}")
            continue



🚀 Iniciando simulación para MSFT [Contexto: 1250]...
   ► Velas Disponibles: 2515 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1250)


Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando itera

✅ Éxito. Guardadas 104 iteraciones para MSFT en context_optimization_db.csv.

🚀 Iniciando simulación para XLF [Contexto: 1250]...
   ► Velas Disponibles: 2515 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1250)


Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando itera

✅ Éxito. Guardadas 104 iteraciones para XLF en context_optimization_db.csv.

🚀 Iniciando simulación para C [Contexto: 1250]...
   ► Velas Disponibles: 2515 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1250)


...

✅ Éxito. Guardadas 161 iteraciones para BTC-USD en context_optimization_db.csv.

🚀 Iniciando simulación para SPY [Contexto: 1250]...
   ► Velas Disponibles: 2515 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1250)


Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando itera

✅ Éxito. Guardadas 104 iteraciones para SPY en context_optimization_db.csv.

🚀 Iniciando simulación para AAPL [Contexto: 1250]...
   ► Velas Disponibles: 2515 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1250)


Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando itera

✅ Éxito. Guardadas 104 iteraciones para AAPL en context_optimization_db.csv.


### 3. Sanidad del Dataset
Lectura rápida para asegurar que la DB se está llenando correctamente.

In [8]:
if os.path.exists(db_path):
    df_final = pd.read_csv(db_path)
    print(f"📊 Total Filas en la Base de Datos: {len(df_final)}")
    display(df_final.head())
    display(df_final.tail())
else:
    print("El archivo CSV aún no ha sido creado.")

📊 Total Filas en la Base de Datos: 681


,Ticker,Periodo_Historia,Intervalo_Velas,Total_Velas_Disponible,Context_Window,Iteracion (Velas Vistas),Drift (k),SINDy R2,Validez,MAPE_B1,...,Hit_B58,CumHit_B58,MAPE_B59,Naive_MAPE_B59,Hit_B59,CumHit_B59,MAPE_B60,Naive_MAPE_B60,Hit_B60,CumHit_B60
0,MSFT,10y,1d,2515,1250,150,0.3,0.165118,OK,2.32,...,0.0,1.0,inf,33.72,0.0,1.0,inf,33.70,0.0,1.0
1,MSFT,10y,1d,2515,1250,170,0.1,0.457711,OK,1.37,...,0.0,1.0,inf,31.65,0.0,1.0,inf,31.46,0.0,1.0
2,MSFT,10y,1d,2515,1250,190,0.8,0.480164,OK,0.48,...,1.0,1.0,12.06,33.19,0.0,1.0,11.7,33.23,0.0,1.0
3,MSFT,10y,1d,2515,1250,210,0.1,0.505090,OK,0.23,...,1.0,0.0,99.99,34.79,0.0,0.0,100.0,35.61,0.0,0.0
4,MSFT,10y,1d,2515,1250,230,0.3,0.513221,OK,3.06,...,1.0,0.0,100.00,36.73,0.0,0.0,100.0,35.35,1.0,0.0


,Ticker,Periodo_Historia,Intervalo_Velas,Total_Velas_Disponible,Context_Window,Iteracion (Velas Vistas),Drift (k),SINDy R2,Validez,MAPE_B1,...,Hit_B58,CumHit_B58,MAPE_B59,Naive_MAPE_B59,Hit_B59,CumHit_B59,MAPE_B60,Naive_MAPE_B60,Hit_B60,CumHit_B60
676,AAPL,10y,1d,2515,1250,2130,1.7,0.984693,OK,0.94,...,0.0,1.0,95.37,14.82,1.0,1.0,105.48,12.63,0.0,1.0
677,AAPL,10y,1d,2515,1250,2150,1.7,0.911912,OK,0.80,...,1.0,1.0,302.16,12.25,0.0,1.0,346.48,7.47,0.0,1.0
678,AAPL,10y,1d,2515,1250,2170,1.7,0.713488,OK,3.71,...,0.0,1.0,275.23,5.89,0.0,1.0,304.62,3.00,0.0,1.0
679,AAPL,10y,1d,2515,1250,2190,1.7,0.863517,OK,1.32,...,1.0,1.0,71.94,10.53,1.0,1.0,72.07,12.27,1.0,1.0
680,AAPL,10y,1d,2515,1250,2210,1.3,0.825543,OK,6.49,...,1.0,1.0,4.63,18.00,1.0,1.0,7.92,21.16,1.0,1.0
